<a href="https://colab.research.google.com/github/AHMADAMJAD0/NLP_with_Olga/blob/main/NLP_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.sparse import lil_matrix, csr_matrix

In [2]:
file_path = "text_news.txt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
    raw_data = file.read()

print("Total characters:", len(raw_data))
print(raw_data[:1000])

Total characters: 7018110

@@3000041 <p> Basketball United States Wins Bronze Medal : The United States used another comeback yesterday to take the bronze medal at the world championships in Buenos Aires , beating Puerto Rico by 107-105 in overtime . Kenny Anderson scored 34 points and made two free throws with one second left to send the game into overtime after Puerto Rico led by 8 points with 1 minute 34 seconds to play in the second half . Yugoslavia , which beat the United States in the semifinals , will face the Soviet Union for the gold medal today . This was the fourth straight international tournament in which the United States had failed to win a gold . Any medal for the United States looked out of the question with 1:34 left in regulation when Raymond Gausse 's 3-pointer gave Puerto Rico a 96-88 lead . Then Anderson hit a jumper with 1:21 to play to get the United States within 6 , and Puerto Rico opted for possession rather than foul shots four times in the final 1:19 . Aft

In [3]:
document_blocks = re.split(r'(?=@@\d+)', raw_data)

documents = []

for block in document_blocks:
    block = block.strip()

    if not block:
        continue

    match = re.match(r'@@(\d+)\s*', block)

    if match:
        doc_id = match.group(1)

        text = re.sub(r'^@@\d+\s*', '', block)

        documents.append({
            "document_id": doc_id,
            "raw_text": text
        })

df = pd.DataFrame(documents)

print("Number of documents:", len(df))

df.head()

Number of documents: 871


,document_id,raw_text
0,3000041,<p> Basketball United States Wins Bronze Medal...
1,3000341,<p> In the airy sitting room of the ranch hous...
2,3000641,<p> It was just a normal summer day when Allan...
3,3000841,"<p> There was no way out this time , so Brett ..."
4,3001041,<p> One jump on a practice rink changed the fa...


In [4]:
df.to_csv(
    "news_corpus_raw.csv",
    index=False,
    encoding="utf-8"
)

print("Saved: news_corpus_raw.csv")

Saved: news_corpus_raw.csv


In [5]:
def clean_text(text):

    # ------------------------------------------------
    # 1. Remove paragraph markers such as <p>
    # ------------------------------------------------
    text = re.sub(r'<p>', ' ', text)

    # ------------------------------------------------
    # 2. Remove repeated @ @ @ @ noise
    # ------------------------------------------------
    text = re.sub(r'(?:@\s*){2,}', ' ', text)

    # ------------------------------------------------
    # 3. Remove URLs
    # Handles formats such as:
    # http://...
    # http : //...
    # ------------------------------------------------
    text = re.sub(
        r'http\s*:\s*//\S+',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # ------------------------------------------------
    # 4. Lowercase
    # ------------------------------------------------
    text = text.lower()

    # ------------------------------------------------
    # 5. Keep only letters and spaces
    # ------------------------------------------------
    text = re.sub(r'[^a-z\s]', ' ', text)

    # ------------------------------------------------
    # 6. Remove extra spaces
    # ------------------------------------------------
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [6]:
df["clean_text"] = df["raw_text"].apply(clean_text)

df[["document_id", "clean_text"]].head()

,document_id,clean_text
0,3000041,basketball united states wins bronze medal the...
1,3000341,in the airy sitting room of the ranch house wh...
2,3000641,it was just a normal summer day when allan hou...
3,3000841,there was no way out this time so brett favre ...
4,3001041,one jump on a practice rink changed the face o...


In [7]:
def tokenize(text):
    return re.findall(r'\b[a-z]+\b', text)

df["tokens"] = df["clean_text"].apply(tokenize)

df[["document_id", "tokens"]].head()

,document_id,tokens
0,3000041,"[basketball, united, states, wins, bronze, med..."
1,3000341,"[in, the, airy, sitting, room, of, the, ranch,..."
2,3000641,"[it, was, just, a, normal, summer, day, when, ..."
3,3000841,"[there, was, no, way, out, this, time, so, bre..."
4,3001041,"[one, jump, on, a, practice, rink, changed, th..."


In [8]:
stopwords = {
    "the", "a", "an", "and", "or", "but",
    "is", "are", "was", "were", "be", "been",
    "being", "to", "of", "in", "on", "at",
    "for", "from", "with", "by", "as",
    "that", "this", "these", "those",
    "he", "she", "it", "they", "we", "you",
    "his", "her", "their", "our", "your",
    "i", "me", "my", "him", "them",
    "has", "have", "had", "do", "does",
    "did", "not"
}

In [9]:
stopwords.remove("not")

In [10]:
def remove_stopwords(tokens):
    return [
        word
        for word in tokens
        if word not in stopwords
        and len(word) > 1
    ]

In [11]:
df["normalized_tokens"] = df["tokens"].apply(remove_stopwords)

df[["document_id", "normalized_tokens"]].head()

,document_id,normalized_tokens
0,3000041,"[basketball, united, states, wins, bronze, med..."
1,3000341,"[airy, sitting, room, ranch, house, where, liv..."
2,3000641,"[just, normal, summer, day, when, allan, houst..."
3,3000841,"[there, no, way, out, time, so, brett, favre, ..."
4,3001041,"[one, jump, practice, rink, changed, face, uni..."


In [12]:
df["normalized_text"] = df["normalized_tokens"].apply(
    lambda words: " ".join(words)
)

print(df["normalized_text"].iloc[0][:1000])

basketball united states wins bronze medal united states used another comeback yesterday take bronze medal world championships buenos aires beating puerto rico overtime kenny anderson scored points made two free throws one second left send game into overtime after puerto rico led points minute seconds play second half yugoslavia which beat united states semifinals will face soviet union gold medal today fourth straight international tournament which united states failed win gold any medal united states looked out question left regulation when raymond gausse pointer gave puerto rico lead then anderson hit jumper play get united states within puerto rico opted possession rather than foul shots four times final after backcourt violation anderson made two free throws seconds left todd day scored seconds later another puerto rico turnover jose ortiz who points rebounds tried inbound ball overthrew federico lopez americans possession chance third comeback victory tournament anderson took bal

In [13]:
df[
    ["document_id", "normalized_text"]
].to_csv(
    "news_corpus_normalized.csv",
    index=False,
    encoding="utf-8"
)

print("Saved: news_corpus_normalized.csv")

Saved: news_corpus_normalized.csv


In [14]:
all_tokens = []

for tokens in df["normalized_tokens"]:
    all_tokens.extend(tokens)

print("Total tokens:", len(all_tokens))

Total tokens: 706734


In [15]:
frequency_distribution = Counter(all_tokens)

print(
    frequency_distribution.most_common(30)
)

[('said', 5385), ('not', 3837), ('who', 3603), ('one', 3035), ('will', 2966), ('about', 2926), ('more', 2647), ('when', 2584), ('all', 2474), ('there', 2449), ('up', 2350), ('would', 2264), ('year', 2229), ('out', 2220), ('new', 2167), ('if', 2148), ('says', 2129), ('what', 2114), ('can', 2060), ('people', 1973), ('so', 1957), ('no', 1925), ('than', 1875), ('two', 1842), ('like', 1824), ('its', 1803), ('after', 1799), ('which', 1770), ('some', 1767), ('years', 1766)]


In [16]:
freq_df = pd.DataFrame(
    frequency_distribution.most_common(),
    columns=["word", "frequency"]
)

freq_df.head(20)

,word,frequency
0,said,5385
1,not,3837
2,who,3603
3,one,3035
4,will,2966
5,about,2926
6,more,2647
7,when,2584
8,all,2474
9,there,2449


In [17]:
vocabulary = sorted(set(all_tokens))

print("Vocabulary size:", len(vocabulary))

print(vocabulary[:50])

Vocabulary size: 45516
['aa', 'aaa', 'aaaa', 'aaaaa', 'aah', 'aam', 'aaron', 'aarp', 'aau', 'ab', 'aba', 'aback', 'abalone', 'abandon', 'abandoned', 'abandoning', 'abandonment', 'abandons', 'abay', 'abayas', 'abba', 'abbado', 'abbas', 'abbey', 'abbie', 'abbott', 'abbreviated', 'abby', 'abc', 'abdalla', 'abdel', 'abdicated', 'abdication', 'abdo', 'abdomen', 'abdominal', 'abdoulaye', 'abducted', 'abduction', 'abductions', 'abduhl', 'abdul', 'abdullah', 'abdullo', 'abdurajak', 'abdurrahim', 'abdus', 'abe', 'abed', 'abel']


In [18]:
word_to_id = {
    word: index
    for index, word in enumerate(vocabulary)
}

id_to_word = {
    index: word
    for word, index in word_to_id.items()
}

In [19]:
for word in vocabulary[:20]:
    print(word, "->", word_to_id[word])

aa -> 0
aaa -> 1
aaaa -> 2
aaaaa -> 3
aah -> 4
aam -> 5
aaron -> 6
aarp -> 7
aau -> 8
ab -> 9
aba -> 10
aback -> 11
abalone -> 12
abandon -> 13
abandoned -> 14
abandoning -> 15
abandonment -> 16
abandons -> 17
abay -> 18
abayas -> 19


In [20]:
vectorizer = CountVectorizer(
    max_features=5000,
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False
)

bow_matrix = vectorizer.fit_transform(
    df["normalized_text"]
)

bow_vocabulary = vectorizer.get_feature_names_out()

print("BOW matrix shape:", bow_matrix.shape)

BOW matrix shape: (871, 5000)


In [21]:
bow_example = pd.DataFrame(
    bow_matrix[:5, :20].toarray(),
    columns=bow_vocabulary[:20]
)

bow_example

,aa,abandoned,abc,ability,able,aboard,abortion,about,above,abroad,absence,absolutely,abu,abuse,academic,academy,accept,accepted,accepting,access
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,17,0,0,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0


In [22]:
term_document_matrix = bow_matrix.T

print(
    "Term-document matrix shape:",
    term_document_matrix.shape
)

Term-document matrix shape: (5000, 871)


In [23]:
WINDOW_SIZE = 4

In [24]:
word_contexts = defaultdict(Counter)

for tokens in df["normalized_tokens"]:

    for i, target_word in enumerate(tokens):

        left_start = max(0, i - WINDOW_SIZE)

        right_end = min(
            len(tokens),
            i + WINDOW_SIZE + 1
        )

        context_words = (
            tokens[left_start:i]
            +
            tokens[i+1:right_end]
        )

        for context_word in context_words:

            if context_word != target_word:
                word_contexts[target_word][context_word] += 1

In [25]:
print(
    word_contexts["president"].most_common(20)
)

[('said', 110), ('vice', 103), ('bush', 66), ('clinton', 42), ('not', 41), ('who', 39), ('former', 39), ('says', 39), ('executive', 23), ('when', 21), ('board', 21), ('obama', 21), ('mr', 20), ('association', 19), ('new', 19), ('under', 18), ('first', 18), ('trump', 18), ('will', 17), ('one', 16)]


In [26]:
def show_context(word, topn=20):

    if word not in word_contexts:
        print("Word not found.")
        return

    for context_word, frequency in \
            word_contexts[word].most_common(topn):

        print(
            f"{context_word:20s}",
            frequency
        )

In [27]:
show_context("doctor")

who                  14
said                 13
if                   10
not                  8
what                 7
would                6
when                 5
told                 5
want                 5
says                 5
there                4
patient              4
out                  4
one                  4
still                4
another              4
could                4
medical              4
chinese              4
something            3


In [28]:
show_context("president")

said                 110
vice                 103
bush                 66
clinton              42
not                  41
who                  39
former               39
says                 39
executive            23
when                 21
board                21
obama                21
mr                   20
association          19
new                  19
under                18
first                18
trump                18
will                 17
one                  16


In [29]:
MIN_FREQUENCY = 30
MAX_VOCAB = 5000

In [30]:
selected_words = [
    word
    for word, freq
    in frequency_distribution.most_common()
    if freq >= MIN_FREQUENCY
]

selected_words = selected_words[:MAX_VOCAB]

print(
    "Selected vocabulary:",
    len(selected_words)
)

Selected vocabulary: 3895
